# Lab 58: The Probability Blindness

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 7, *The Decision Bias*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-58.ipynb)

**What you will do:** tally your own cups-and-coin games, run a large Monte Carlo simulation to confirm the switch and stay win rates, pool the class's tallies, and test whether a small language model can tell a genuine Monty Hall problem from a look-alike where switching does not help.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 58 you played the **Monty Hall problem** with three cups and a coin: you pick a cup, the
host (who knows where the coin is) reveals an empty cup among the other two, and you decide
whether to switch or stay. Switching wins about two-thirds of the time; staying wins about
one-third. The host's choice is not random — it carries information, and that information
favours the cup you did not pick.

## Record your results

Enter your own tally from playing 30 games (alternating switch and stay), or from the
montyhall.io simulator.

In [ ]:
tally = pd.DataFrame({
    "strategy": ["switch", "stay"],
    "wins": [10, 5],      # example data: replace with your own tally
    "games": [15, 15],    # example data
})
tally["win_rate"] = tally["wins"] / tally["games"]
display(tally)
tally.plot.bar(x="strategy", y="win_rate", legend=False, color=["steelblue", "grey"], figsize=(4.5, 3))
plt.axhline(2 / 3, color="steelblue", linestyle="--", linewidth=1)
plt.axhline(1 / 3, color="grey", linestyle="--", linewidth=1)
plt.ylabel("win rate"); plt.ylim(0, 1); plt.xticks(rotation=0); plt.show()

## Compare

The lab states that switching wins about **2/3** of the time and staying wins about **1/3**.
Before comparing your own small tally (noisy with only 15 games per strategy) against that
figure, run a large simulation of the game itself, which the lab describes as available
online (e.g. montyhall.io) after 1,000 trials.

In [ ]:
def play_monty_hall(switch, n_trials, rng):
    prize = rng.integers(0, 3, n_trials)
    pick = rng.integers(0, 3, n_trials)
    wins = np.zeros(n_trials, dtype=bool)
    for i in range(n_trials):
        # the host can only open a cup that is neither your pick nor the prize
        candidates = [c for c in range(3) if c != pick[i] and c != prize[i]]
        host_opens = rng.choice(candidates)
        if switch:
            final = next(c for c in range(3) if c not in (pick[i], host_opens))
        else:
            final = pick[i]
        wins[i] = final == prize[i]
    return wins.mean()

rng = np.random.default_rng(2026)
sim_switch = play_monty_hall(True, 10000, rng)
sim_stay = play_monty_hall(False, 10000, rng)
print(f"Simulation (10,000 trials): switch wins {sim_switch:.1%}, stay wins {sim_stay:.1%}")
print(f"Lab's stated values:        switch ~66.7% (2/3),  stay ~33.3% (1/3)")
print(f"Your own small tally:       switch {tally.set_index('strategy').loc['switch', 'win_rate']:.0%}, "
      f"stay {tally.set_index('strategy').loc['stay', 'win_rate']:.0%} (only {tally.games.iloc[0]} games each -- expect noise)")

## The AI side

Most people's intuition says "two cups left, so 50-50" — the lab calls this the brain's
failure of conditional probability. Below, a small open instruction-following model
(flan-t5-small) is asked the standard problem, and then a look-alike variant, the
**Monty Fall** problem, where the host does not know where the coin is and just happens to
reveal an empty cup by luck. In Monty Fall, the host's action carries no information, so
switching and staying are genuinely 50-50 — a model that always says "switch" regardless of
which version it is asked is pattern-matching a famous puzzle, not reasoning about it.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
from transformers import pipeline

qa = pipeline("text2text-generation", model="google/flan-t5-small")

standard = (
    "You pick one of three cups. The host, who knows which cup hides the coin, lifts one of "
    "the other two cups and shows it is empty. Should you switch to the remaining cup or stay "
    "with your first choice? Answer switch or stay and say why in one sentence."
)
monty_fall = (
    "You pick one of three cups. The host does not know which cup hides the coin. The host "
    "lifts one of the other two cups at random, and it happens to be empty. Should you switch "
    "to the remaining cup or stay with your first choice? Answer switch or stay and say why in "
    "one sentence."
)
for label, prompt in [("Standard Monty Hall", standard), ("Monty Fall (host is ignorant)", monty_fall)]:
    answer = qa(prompt, max_new_tokens=60)[0]["generated_text"]
    print(f"{label}:\n  {answer}\n")

The correct answers differ: switch in the standard version, but it makes no difference in
Monty Fall, because the host's lucky, uninformed reveal carries no evidence about where the
coin is. flan-t5-small is a small model and often cannot make this distinction reliably — it
may recite "switch" for both because the Monty Hall problem is so heavily represented in its
training data. That is the point of the demo: fluent, confident text is not the same as
correct probabilistic reasoning, in models or in people.

## Pool the class data

Pool switch/stay tallies from the class (anonymous IDs, not names).

In [ ]:
import io
csv_text = """id,strategy,wins,games
P01,switch,10,15
P01,stay,5,15
P02,switch,9,15
P02,stay,6,15
P03,switch,11,15
P03,stay,4,15
"""  # example data: replace with your class CSV, e.g. pd.read_csv("montyhall_class.csv")
df = pd.read_csv(io.StringIO(csv_text))
summary = df.groupby("strategy")[["wins", "games"]].sum()
summary["win_rate"] = summary.wins / summary.games
display(summary)
boot = {s: [df.loc[df.strategy == s].sample(frac=1, replace=True).wins.sum()
            / df.loc[df.strategy == s].sample(frac=1, replace=True).games.sum()
            for _ in range(3000)] for s in ["switch", "stay"]}
for s in ["switch", "stay"]:
    lo, hi = np.percentile(boot[s], [2.5, 97.5])
    print(f"{s}: pooled win rate {summary.loc[s, 'win_rate']:.0%} (95% bootstrap interval {lo:.0%}-{hi:.0%})")
print("Lab's stated values: switch ~67%, stay ~33%")

## Questions to think about

1. How close was your own 30-game tally to the lab's 2/3-vs-1/3 figures? Was it closer once you pooled with the class?
2. The 10,000-trial simulation should land very close to 66.7%/33.3%. Why does a simulation with more trials get closer to the true probability than your own 15-game tally, even though both use the same rules?
3. In the AI demo, did flan-t5-small give a different answer for Monty Fall than for the standard problem? What would a genuinely reasoning system need to track to get both right?
4. The lab says pigeons learn to switch more reliably than humans through repeated experience. Does your own switch-win-rate improve if you look only at your later games versus your earlier ones?

## Challenge

Extend the simulation to a version with 100 cups instead of 3, where the host opens 98 empty
cups after your pick. Run it and check whether the switching win rate matches the intuition the
lab describes ("the chance you picked correctly was only 1/100").

In [ ]:
def play_monty_hall_n(n_cups, switch, n_trials, rng):
    prize = rng.integers(0, n_cups, n_trials)
    pick = rng.integers(0, n_cups, n_trials)
    wins = (pick == prize) if not switch else (pick != prize)
    return wins.mean()

rng = np.random.default_rng(2026)
n_cups = 100   # example: change this to test other cup counts
sim_switch = play_monty_hall_n(n_cups, True, 20000, rng)
sim_stay = play_monty_hall_n(n_cups, False, 20000, rng)
print(f"With {n_cups} cups: switch wins {sim_switch:.1%} (expected ~{(n_cups - 1) / n_cups:.1%}), "
      f"stay wins {sim_stay:.1%} (expected ~{1 / n_cups:.1%})")
print("With more cups the advantage of switching becomes overwhelming, even though the logic")
print("is identical to the 3-cup version.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-58.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")